
# Set C — Electrophysiology Fundamentals (LEGO‑Colab)

**Final (Option 1 style)** — 2026-03-09  \
**Author:** Satish Nair

> If you did **not** run Set B in this notebook, run the Starter below once. Otherwise, skip it.  
> Each section includes a short **systems‑theoretic callout** and a **Try with Copilot** prompt.

---
## Table of Contents
- [🔰 Colab Starter](#starter)
- [C1. What an intracellular electrode measures](#c1)
- [C2. Current clamp — inject I, observe V(t)](#c2)
- [C3. Voltage clamp — control V, measure I](#c3)
- [C4. Liquid junction potentials (LJP) — offset & correction](#c4)
- [C5. Series/access resistance (Rs) — clamp error](#c5)
- [C6. Bridge balance & DCC (conceptual)](#c6)
- [C7. Estimate R_in and τ](#c7)
- [C8. I–V curve & E_rev](#c8)
- [C9. Threshold, rheobase, refractory](#c9)
- [C10. Model‑ready metadata capture](#c10)
---


### 🔰 Colab Starter (run once if starting fresh) <a id='starter'></a>

In [ ]:

!pip -q install neuron==8.2.4
try:
    from neuron import h, gui
except Exception:
    from neuron import h
from neuron.units import ms, mV
import matplotlib.pyplot as plt

h.load_file("stdrun.hoc")
soma = h.Section(name='soma')
soma.L = 20; soma.diam = 20; soma.Ra = 100
soma.insert('hh')
soma.insert('pas'); soma.g_pas=0.0002; soma.e_pas=-65; soma.cm=1.0

t = h.Vector().record(h._ref_t)
v = h.Vector().record(soma(0.5)._ref_v)

def run(sim_ms=60, v_init=-65, title=""):
    h.finitialize(v_init * mV)
    h.continuerun(sim_ms * ms)
    plt.figure(figsize=(7,3.3))
    plt.plot(t, v, 'k')
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title(title)
    plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()

print("Set C starter ready.")


## C1. What an intracellular electrode measures <a id='c1'></a>

In [ ]:

for v0 in [-75, -65, -55]:
    run(sim_ms=40, v_init=v0, title=f"C1: Baseline at init V={v0} mV (no injection)")


> **Systems callout — Measurement reality**  
- Intracellular electrodes report membrane potential V relative to bath/ground.
- Resting V depends on ionic gradients & leak reversal; initialization mimics different baseline states.

**Try with Copilot**  
> Explain why two cells might show different resting potentials even under identical recording solutions.

## C2. Current clamp — inject I, observe V(t) <a id='c2'></a>

In [ ]:

stim = h.IClamp(soma(0.5))
stim.delay = 5
stim.dur   = 50
for A in [-0.05, 0.05, 0.10, 0.15]:
    stim.amp = A
    run(sim_ms=70, title=f"C2: Current clamp, amp={A} nA")


> **Systems callout — I→V transfer**  
- Current clamp reveals passive τ and nonlinear threshold behaviors.
- Amplitude & duration jointly determine depolarization and spike probability.

**Try with Copilot**  
> Propose a pulse that is subthreshold by ~2 mV margin and justify from your earlier traces.

## C3. Voltage clamp — control V, measure I <a id='c3'></a>

In [ ]:

vclamp = h.VClamp(soma(0.5))
vclamp.dur[0] = 10; vclamp.amp[0] = -80
vclamp.dur[1] = 20; vclamp.amp[1] = -20
vclamp.dur[2] = 20; vclamp.amp[2] =  40

i_vc = h.Vector().record(vclamp._ref_i)
t_vc = h.Vector().record(h._ref_t)

h.finitialize(-65*mV); h.continuerun(50*ms)
plt.figure(figsize=(7,4))
plt.subplot(2,1,1); plt.plot(t_vc, v, 'k'); plt.ylabel('V (mV)'); plt.title('C3: Voltage clamp protocol')
plt.subplot(2,1,2); plt.plot(t_vc, i_vc, 'C1'); plt.xlabel('Time (ms)'); plt.ylabel('I (nA)')
plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — V→I mapping**  
- Voltage clamp isolates ionic currents as a function of command V.
- Useful to derive I–V relationships and infer reversal potentials.

**Try with Copilot**  
> Identify which step produced the largest inward current and explain why in terms of driving force.

## C4. Liquid junction potentials (LJP) — offset & correction <a id='c4'></a>

In [ ]:

LJP_offset = +7.0
stim.amp = 0.0
h.finitialize(-65*mV); h.continuerun(40*ms)
measured = [vv + LJP_offset for vv in v]
plt.figure(figsize=(7,3.3))
plt.plot(t, v, label='True V')
plt.plot(t, measured, '--', label=f'Measured V (+{LJP_offset} mV LJP)')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title('C4: LJP bias demo')
plt.legend(); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()

corrected = [vm - LJP_offset for vm in measured]
plt.figure(figsize=(7,3.3))
plt.plot(t, corrected, 'C2', label='Corrected V')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title('C4: After LJP correction')
plt.legend(); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Reference offsets**  
- LJP arises at interfaces of dissimilar solutions; it biases measured V.
- Apply an additive correction to recover the true membrane potential.

**Try with Copilot**  
> Given two solution recipes, estimate the expected LJP sign and magnitude qualitatively.

## C5. Series/access resistance (Rs) — clamp error illustration <a id='c5'></a>

In [ ]:

import numpy as np
vclamp = h.VClamp(soma(0.5))
vclamp.dur[0] = 10; vclamp.amp[0] = -80
vclamp.dur[1] = 30; vclamp.amp[1] =  0
vclamp.dur[2] = 10; vclamp.amp[2] = -80

i_vc = h.Vector().record(vclamp._ref_i)
t_vc = h.Vector().record(h._ref_t)

h.finitialize(-65*mV); h.continuerun(50*ms)
cmd = np.array([-80 if tt<10 else (0 if tt<40 else -80) for tt in t_vc])
for Rs in [5, 20, 50]:
    v_actual = cmd - np.array(i_vc)*Rs
    plt.figure(figsize=(7,3.3))
    plt.plot(t_vc, cmd, label='V_cmd')
    plt.plot(t_vc, v_actual, label=f'V_actual (Rs={Rs} MΩ)')
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title('C5: Series resistance reduces clamp fidelity')
    plt.legend(); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Voltage error**  
- Finite Rs produces a drop proportional to current: V_actual ≈ V_cmd − I·Rs.
- Large Rs distorts activation curves and time courses in voltage clamp.

**Try with Copilot**  
> Suggest a maximum acceptable Rs for your protocol and justify using expected current amplitudes.

## C6. Bridge balance & DCC (conceptual) <a id='c6'></a>

In [ ]:

stim.amp = 0.2; stim.delay=5; stim.dur=20
h.finitialize(-65*mV); h.continuerun(40*ms)
raw_v = list(v)
Relec = 30
I_trace = [stim.amp if (5<=tt<=25) else 0.0 for tt in t]
corrected_v = [rv - i*Relec for rv,i in zip(raw_v, I_trace)]
plt.figure(figsize=(7,3.3))
plt.plot(t, raw_v, label='Raw Vm')
plt.plot(t, corrected_v, label=f'Bridge-corrected (Relec={Relec} MΩ)')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title('C6: Bridge balance concept (didactic)')
plt.legend(); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Subtracting electrode drop**  
- Bridge balance compensates I·Relec from the measured Vm under current clamp.
- Discontinuous current clamp (DCC) alternates inject/sample to reduce artifacts.

**Try with Copilot**  
> Explain how mis‑setting Relec biases R_in estimates, and how DCC mitigates this.

## C7. Estimate R_in and τ <a id='c7'></a>

In [ ]:

stim.amp = -0.03; stim.delay=5; stim.dur=30
h.finitialize(-65*mV); h.continuerun(50*ms)
import numpy as np

t_np = np.array(t); v_np = np.array(v)
mask_ss = (t_np>33) & (t_np<38)
dV = v_np[mask_ss].mean() - (-65)
Rin_MOhm = abs(dV)/abs(stim.amp)
print(f"Estimated R_in ≈ {Rin_MOhm:.1f} MΩ (rough)")

v0 = -65; vss = -65 + dV; target = v0 + 0.632*(vss - v0)
idx = np.where(v_np >= target)[0]
tau_est = (t_np[idx[0]] - 5) if len(idx)>0 else float('nan')
print(f"Estimated tau ≈ {tau_est:.2f} ms")


> **Systems callout — From steps to parameters**  
- R_in ≈ ΔV/ΔI from steady segment; τ from 63% rise/decay timing.
- C = τ/R and G = 1/R link electrophysiology to circuit parameters.

**Try with Copilot**  
> Compare τ from rising vs falling phases; explain discrepancies (e.g., inward rectification).

## C8. I–V curve & E_rev <a id='c8'></a>

In [ ]:

steps = [-90,-70,-50,-30,-10,10,30,50]
I_ss = []
for Vc in steps:
    vcl = h.VClamp(soma(0.5))
    vcl.dur[0]=5; vcl.amp[0]=-80
    vcl.dur[1]=30; vcl.amp[1]=Vc
    vcl.dur[2]=5; vcl.amp[2]=-80
    i_vec = h.Vector().record(vcl._ref_i)
    t_vec = h.Vector().record(h._ref_t)
    h.finitialize(-65*mV); h.continuerun(40*ms)
    t_np = list(t_vec); i_np = list(i_vec)
    idx = [i for i,tt in enumerate(t_np) if 30<=tt<=34]
    I_ss.append(sum(i_np[i] for i in idx)/len(idx))

import matplotlib.pyplot as plt
plt.figure(figsize=(5,4))
plt.plot(steps, I_ss, 'o-')
plt.axhline(0, color='k', lw=1)
plt.xlabel('Command V (mV)'); plt.ylabel('Steady I (nA)'); plt.title('C8: I–V curve')
plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — E_rev from I–V**  
- Where I crosses 0 gives the apparent reversal potential under the protocol.
- Slope around a point reflects local conductance; nonlinearities reveal channel gating.

**Try with Copilot**  
> Estimate E_rev from your plot and discuss the biophysical interpretation.

## C9. Threshold, rheobase, refractory <a id='c9'></a>

In [ ]:

amps = [0.02,0.04,0.06,0.08,0.10,0.12]
stim.delay=5; stim.dur=100
for A in amps:
    stim.amp=A
    run(sim_ms=120, title=f"C9: Staircase rheobase search, A={A} nA")

stim1 = h.IClamp(soma(0.5)); stim1.dur=2; stim1.amp=0.2
stim2 = h.IClamp(soma(0.5)); stim2.dur=2; stim2.amp=0.2
for isi in [2,4,6,8,10,12]:
    stim1.delay=5
    stim2.delay=5+isi
    run(sim_ms=25, title=f"C9: Paired pulses, ISI={isi} ms")


> **Systems callout — Excitability metrics**  
- Rheobase: minimal sustained current to elicit a spike.
- Refractory: time‑dependent recovery of excitability after a spike.

**Try with Copilot**  
> From your staircase, report a rheobase estimate and compare to predictions from τ and R_in.

## C10. Model‑ready metadata capture <a id='c10'></a>

In [ ]:

experiment_meta = {
    "internal_solution_mM": {"K":140, "Na":10, "Cl":5, "Mg":1, "ATP":4},
    "external_solution_mM": {"Na":140, "K":3, "Cl":140, "Ca":2, "Mg":1},
    "temperature_C": 34,
    "LJP_mV": 7.0,
    "Rs_MOhm": 20,
    "cap_comp": "fast 70%, slow 30%",
    "filters": {"lowpass_Hz": 10000, "sampling_kHz": 20},
    "holding_potential_mV": -65
}
experiment_meta


> **Systems callout — Reproducibility**  
- Capturing metadata (solutions, T, LJP, Rs, filtering) enables comparisons and model fitting.
- Encourage students to log these fields alongside each protocol.

## Reflection
- When would you prefer current clamp vs voltage clamp given your experimental goal?
- Which sources of error (LJP, Rs, bridge) dominate in your setup, and how would you quantify them?
- How do R_in and τ estimates inform expectations about threshold (C9)?